In [1]:
from fastai.vision.all import *
import torch.nn.functional as F
import numpy as np, pandas as pd, os, sys, re, random

np.set_printoptions(threshold=sys.maxsize)



/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
df = pd.read_csv(
    "../input/rsna-miccai-brain-tumor-radiogenomic-classification/train_labels.csv",
    header=0,
    names=["id", "value"],
    dtype=object,
)
df = df[~df.id.isin(["00109", "00123", "00709"])]




In [3]:
def natural_sort(l):
    convert = lambda text: int(text) if text.isdigit() else text.lower()
    alphanum_key = lambda key: [convert(c) for c in re.split("([0-9]+)", key)]
    return sorted(l, key=alphanum_key)




In [4]:
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
from tqdm import tqdm
from PIL import Image

INPUT = "../input/rsna-miccai-brain-tumor-radiogenomic-classification"

os.makedirs("./train", exist_ok=True)
os.makedirs("./test", exist_ok=True)


def process_dicom(path, outpath):
    dicom = pydicom.dcmread(path)  # <-- fixed
    data = apply_voi_lut(dicom.pixel_array, dicom)
    if dicom.PhotometricInterpretation == "MONOCHROME1":
        data = np.amax(data) - data
    data = data - np.min(data)
    denom = np.max(data)
    if denom == 0:  # avoid div‑by‑zero
        denom = 1e-6
    data = data / denom
    data = (data * 255).astype(np.uint8)
    height, width = data.shape
    image_out = Image.fromarray(data, mode="L")
    image_out.save(outpath)


def get_dicom_files(input_dir, dataset="train"):
    for subdir, dirs, files in os.walk(f"{input_dir}/{dataset}"):
        if len(files) == 0:
            continue
        filename = natural_sort(files)[len(files) // 2]  # middle slice
        filepath = os.path.join(subdir, filename)
        if filepath.lower().endswith(".dcm") and "FLAIR" in filepath:
            cur_id = subdir.split("/")[-2]
            outpath = os.path.join(f"./{dataset}", f"{cur_id}.png")
            process_dicom(filepath, outpath)


get_dicom_files(INPUT, "train")
get_dicom_files(INPUT, "test")



/tmp/ipykernel_55/653501655.py:24: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image_out = Image.fromarray(data, mode="L")


In [5]:
df["file"] = "./train/" + df["id"].astype(str) + ".png"
df = df[df["file"].apply(os.path.isfile)].reset_index(drop=True)



In [6]:
# keep as string categories; FastAI will handle conversion
df["value"] = df["value"].astype(str)



In [7]:
dls = ImageDataLoaders.from_df(
    df,
    path=".",
    fn_col="file",
    label_col="value",
    valid_pct=0.2,
    seed=42,
    item_tfms=Resize(224),
    bs=64,
)




In [8]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # input images are grayscale -> 1 channel
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 2)  # 2 classes for binary classification

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x




In [9]:
print(Net())



Net(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=2, bias=True)
)


In [10]:
learn = Learner(
    dls,
    Net(),
    loss_func=CrossEntropyLossFlat(),
    metrics=[error_rate, accuracy],
    path="/tmp/model/",
).to_fp16()



In [11]:
learn.fine_tune(5, base_lr=1e-2)  # 5 epochs are sufficient for a baseline



epoch,train_loss,valid_loss,error_rate,accuracy,time


RuntimeError: Given groups=1, weight of size [6, 1, 5, 5], expected input[64, 3, 224, 224] to have 1 channels, but got 3 channels instead

In [12]:
learn.show_results()



RuntimeError: Given groups=1, weight of size [6, 1, 5, 5], expected input[64, 3, 224, 224] to have 1 channels, but got 3 channels instead

In [13]:
test_ids = [
    name
    for name in os.listdir(
        "../input/rsna-miccai-brain-tumor-radiogenomic-classification/test/"
    )
    if os.path.isdir(
        os.path.join(
            "../input/rsna-miccai-brain-tumor-radiogenomic-classification/test/", name
        )
    )
]
df_test = pd.DataFrame({"id": test_ids})
df_test["file"] = "./test/" + df_test["id"] + ".png"
# keep only existing png files
df_test = df_test[df_test["file"].apply(os.path.isfile)].reset_index(drop=True)



In [14]:
preds = []
for fp in df_test["file"]:
    try:
        pred = learn.predict(fp)
        # pred[2] is a tensor of probabilities for the two classes
        prob = float(pred[2][1].item())  # probability of class "1"
    except Exception as e:
        prob = 0.0  # fallback if prediction fails
    preds.append(prob)

df_test["value"] = preds



In [15]:
submission = df_test.rename(columns={"id": "BraTS21ID", "value": "MGMT_value"})
submission.to_csv("submission.csv", index=False)